### 벡터 DB
- 문서가 많을 때 대량 검색을 알아서 해주는 벡터DB 구축
- 인덱싱(색인), 100만개 정도 되더라도 검색 및 계산이 거의 사라짐
- 메타 데이터(카테고리,날짜) 바탕ㅇ로 검색이 가능

질문(qeustion) -> retrever(검색기)-벡터DB -> Generator(응답생성)

- 근사 검색(ANN)
- HNSW
    - 벡터들을 여러 층의 그래프로 연결해 두고 위에서 대충 방향 잡고 아래로 내려가면서 데이터를 훌튼 방식

### 벡터 DB 종류
- ChromaDB
- FAISS - 옛날 로컬 DB 벡터
- Qdrant
- Pinecone
- pgvector
- neo4j(그래프DB) - 그냥 벡터도 가능
- milvus -  데이터가 엄청 많을 때

In [1]:
import chromadb

print(chromadb.__version__)

1.5.9


In [2]:
import pandas as pd
df = pd.read_csv("../data/11-1_뉴스정제.csv").head(200).reset_index(drop=True)
df.head()

,제목,본문,카테고리,요약,출처URL,정제본문
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,경제,"6 6일 현대백화점그룹이 광주시에 문화복합몰을 만든다고 6일 밝혔으며, 광주시는 서...",https://n.news.naver.com/mnews/article/001/001...,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,전주 뉴시스 김얼 기자 이스타항공 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직...,경제,이이스항공은 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직 전 의원이 출소한 것...,https://n.news.naver.com/mnews/article/003/001...,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 ‘10주년 기념주...,경제,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 소셜미디어 인스타...,https://n.news.naver.com/mnews/article/366/000...,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...
3,오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은,img tag s 지난 30일 서울의 한 주유소. 〈사진 연합뉴스〉 img tag ...,경제,정부는 고유가 상황에 따라 국민의 유류비 부담 완화를 위해 이날부터 유류세를 법정 ...,https://n.news.naver.com/mnews/article/437/000...,img tag s 지난 30일 서울의 한 주유소 사진 연합뉴스 img tag e 오...
4,푸르덴셜생명 더 큰 드림 변액연금보험Ⅱ에 신규펀드 13종 추가,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...,경제,지난르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림 변액연금보...,https://n.news.naver.com/mnews/article/014/000...,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...


In [3]:
docs = df['정제본문'].tolist()

print(len(docs))

200


In [4]:
# 메타데이터로 쓸만한 내용 - 나중에는 LLM 으로 필요 메타데이터 추출 -> 전처리
df[['제목', '카테고리', '출처URL']].head()

,제목,카테고리,출처URL
0,현대백화점그룹 더현대 광주 추진,경제,https://n.news.naver.com/mnews/article/001/001...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,경제,https://n.news.naver.com/mnews/article/003/001...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,경제,https://n.news.naver.com/mnews/article/366/000...
3,오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은,경제,https://n.news.naver.com/mnews/article/437/000...
4,푸르덴셜생명 더 큰 드림 변액연금보험Ⅱ에 신규펀드 13종 추가,경제,https://n.news.naver.com/mnews/article/014/000...


## Cromadb 컬렉션 만들기

1. 임베딩 함수 
2. 거리척도(코사인유사도)

In [5]:
import os
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction  # chromaDB 에서 Openai 임베딩 모델
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # .env 파일에 저장되어있는 api 키 가져오기
client = OpenAI()

In [6]:
embeddings = OpenAIEmbeddingFunction(
    model_name="text-embedding-3-small"
)

embed_client = chromadb.Client()

# collection 
collection = embed_client.get_or_create_collection(
    name="news_collection",
    embedding_function=embeddings,
    configuration={'hnsw' : {"space" : "cosine"}}
)

In [7]:
print(collection.name)

news_collection


In [8]:
ids = df.index.astype(str).tolist()
metadata = df[['제목', '카테고리', '출처URL']].to_dict('records')
metadata

[{'제목': '현대백화점그룹 더현대 광주 추진',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/001/0013293519?sid=101'},
 {'제목': '이스타항공 이상직 회사와 무관…오해 살 언동 말아야',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/003/0011282043?sid=101'},
 {'제목': '농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/366/0000824966?sid=101'},
 {'제목': '오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/437/0000304023?sid=101'},
 {'제목': '푸르덴셜생명 더 큰 드림 변액연금보험Ⅱ에 신규펀드 13종 추가',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/014/0004860779?sid=101'},
 {'제목': '오늘부터 사고 나면 중대재해처벌법 기소 가능성 더 커져',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/277/0005111594?sid=101'},
 {'제목': '역세권에 신규분양 봇물…천안 부성지구 한라비발디 654가구 분양',
  '카테고리': '경제',
  '출처URL': 'https://n.news.naver.com/mnews/article/374/0000293010?sid=101'},
 {'제목': '식량 대란 오나',
  '카테고리': '경제',
  '출처URL': 'https://n

In [9]:
collection.upsert(ids=ids, documents=docs, metadatas=metadata)
collection.count()

200

In [10]:
# 검색기
question = "대출 정보에 대해서 알려줘"

result = collection.query(query_texts=[question], n_results=5)
result

{'ids': [['111', '51', '120', '130', '3']],
 'embeddings': None,
 'documents': [['보건복지부 제공 보건복지부는 저소득 청년에 저축액의 최대 3배까지 추가 적립해주는 청년내일저축계좌 가입자를 오는 18일부터 모집한다고 밝혔다 해당 제도는 월 10만원을 저축하면 정부가 지원금 월 10만원을 추가 적립하는 방식으로 3년간 지원해 청년의 자산 형성을 도와주는 청년특별대책제도다 참여자는 3년 만기 시 본인 납입액 360만원을 포함해 총 720만원과 예금이자를 수령하게 된다 청년내일저축계좌는 신청 시점 기준 만 19 34세인 청년이 대상이다 본인 소득 가구 소득 가구 재산 등 3가지 기준을 충족해야 한다 신청 당시에 근로 중이어야 하며 근로 사업소득이 50만원 초과 200만원 이하여야 한다 또한 자신이 속한 가구의 소득이 기준 중위소득 100 2022년 4인 가구 기준 512만1080원 이하여야 한다 가구 재산 기준은 지역에 따라 차등이 있다 대도시 3억5000만원 중소도시 2억원 농어촌 1억7000만원 이하인 가구가 대상이다 신청자가 기초생활수급자이거나 차상위계층 기준 중위소득 50 이하 이라면 혜택이 훨씬 크다 참여자가 10만원을 적립할 때마다 정부 지원금 30만원이 지급된다 3년 뒤 만기 때 총 1440만원의 적립금과 예금이자를 수령하게 된다 또한 가입 가능 연령이 만 15 39세로 더 넓으며 근로 사업소득기준도 면제된다 복지부는 그동안 청년층 대상 자산형성지원 사업을 해왔으나 그 대상이 수급자와 차상위계층에만 한정됐었다 그러나 이번에는 중위소득 100 이하인 청년으로 대상이 확대되면서 신청 대상이 지난해 1만8000명에서 올해 10만4000명으로 6배 증가했다 가입을 희망하는 청년은 복지로 홈페이지 www bokjiro go kr 를 통해 신청하면 된다 정부는 원활한 신청을 위해 신청 시작일인 18일부터 2주 7월 18 29일 간은 출생일 기준으로 5부제를 시행한다 5부제 기간에 신청하지 못한 경우 

In [11]:
result['documents']

[['보건복지부 제공 보건복지부는 저소득 청년에 저축액의 최대 3배까지 추가 적립해주는 청년내일저축계좌 가입자를 오는 18일부터 모집한다고 밝혔다 해당 제도는 월 10만원을 저축하면 정부가 지원금 월 10만원을 추가 적립하는 방식으로 3년간 지원해 청년의 자산 형성을 도와주는 청년특별대책제도다 참여자는 3년 만기 시 본인 납입액 360만원을 포함해 총 720만원과 예금이자를 수령하게 된다 청년내일저축계좌는 신청 시점 기준 만 19 34세인 청년이 대상이다 본인 소득 가구 소득 가구 재산 등 3가지 기준을 충족해야 한다 신청 당시에 근로 중이어야 하며 근로 사업소득이 50만원 초과 200만원 이하여야 한다 또한 자신이 속한 가구의 소득이 기준 중위소득 100 2022년 4인 가구 기준 512만1080원 이하여야 한다 가구 재산 기준은 지역에 따라 차등이 있다 대도시 3억5000만원 중소도시 2억원 농어촌 1억7000만원 이하인 가구가 대상이다 신청자가 기초생활수급자이거나 차상위계층 기준 중위소득 50 이하 이라면 혜택이 훨씬 크다 참여자가 10만원을 적립할 때마다 정부 지원금 30만원이 지급된다 3년 뒤 만기 때 총 1440만원의 적립금과 예금이자를 수령하게 된다 또한 가입 가능 연령이 만 15 39세로 더 넓으며 근로 사업소득기준도 면제된다 복지부는 그동안 청년층 대상 자산형성지원 사업을 해왔으나 그 대상이 수급자와 차상위계층에만 한정됐었다 그러나 이번에는 중위소득 100 이하인 청년으로 대상이 확대되면서 신청 대상이 지난해 1만8000명에서 올해 10만4000명으로 6배 증가했다 가입을 희망하는 청년은 복지로 홈페이지 www bokjiro go kr 를 통해 신청하면 된다 정부는 원활한 신청을 위해 신청 시작일인 18일부터 2주 7월 18 29일 간은 출생일 기준으로 5부제를 시행한다 5부제 기간에 신청하지 못한 경우 8월 1 5일에 생일과 무관하게 신청할 수 있다 대상자 선정 결과는 소득 재산 조사 등을 거쳐 10월 중에 발표된다 곽숙영 복지부 복지정책관은

### Qdrant
- 방식은 거의 chromadb

In [12]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import json   # JSON 처리
import os # 파일 존재 여부와 경로 처리
from datetime import datetime # 저장 시간 기록
import pandas as pd # 판다스
from dotenv import load_dotenv # 환경변수 불러오기
from openai import OpenAI # OpenAI 클라이언트


In [13]:
qdrant = QdrantClient(path="./qdrant_news_db")
qdrant

In [36]:
qdrant.create_collection(
    collection_name="news1_qdrant",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)

True

In [ ]:
qdrant.get_collections()


CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=200, segments_count=1, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_op

### 문서 적재
- 문서를 포인트 단위로 넣는 방식
- id, vector, playload(부가정보=메타데이터+원문)

In [24]:
client = OpenAI()

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=docs
)

In [ ]:
response.data[:10]

[Embedding(embedding=[-0.020660400390625, -0.0028591156005859375, 0.034210205078125, 0.034271240234375, 0.01126861572265625, 0.0037384033203125, 0.0038433074951171875, 0.0570068359375, -0.0007305145263671875, -0.059722900390625, -0.034515380859375, -0.0190277099609375, 0.0014085769653320312, -0.04046630859375, 0.0245513916015625, -0.011505126953125, -0.004791259765625, -0.0009984970092773438, 0.0478515625, -0.012451171875, -0.0214691162109375, -0.02880859375, 0.01332855224609375, 0.006488800048828125, -0.007358551025390625, -0.018646240234375, 0.01395416259765625, 0.0479736328125, -0.0010051727294921875, -0.01537322998046875, -0.03497314453125, -0.03692626953125, 0.0318603515625, 0.015472412109375, 0.0149078369140625, 0.0260772705078125, 0.0093994140625, 0.0213623046875, 0.0309295654296875, -0.00678253173828125, 0.03057861328125, -0.006877899169921875, 0.003173828125, 0.004497528076171875, 0.0196990966796875, 0.0259857177734375, -0.0028095245361328125, 0.001773834228515625, 0.053283691

In [25]:
points = []
for i in range(len(docs)):
    points.append(PointStruct(
        id=i,
        vector=response.data[i].embedding,
        payload={'제목' : df['제목'][i], '카테고리' : df['카테고리'][i],"원문":docs[i]}
    ))



In [31]:
qdrant.upsert(collection_name="news_qdrant",
              points=points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [16]:
#임베딩 함수
import numpy as np

# 임베딩 함수 만들기
def embed(texts):
    """텍스트 목록을 받아서 벡터 배열로 변환하는 함수. openai embedding 을 사용해서 문서수 x 1536차원으로 변환"""

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    return np.array([item.embedding for item in response.data])

In [17]:
question = "대출 정보에 대해서 알려줘"

query_vec = embed([question])[0]

result = qdrant.query_points(
    collection_name="news_qdrant",
    query=query_vec.tolist(),
    limit=5
)
print(result)

points=[ScoredPoint(id=111, version=0, score=0.3627533353944139, payload={'제목': '월 10만원 저축하면 정부가 10만원 더…18일부터 청년내일저축계좌 모집', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=51, version=0, score=0.3620797713274341, payload={'제목': '은행 가계대출 반년간 9조 넘게 줄었다', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=120, version=0, score=0.34174196642508886, payload={'제목': '문과생도 IT 인재로 육성…취업률 59% ‘눈길’', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=130, version=0, score=0.3183656206335821, payload={'제목': '뉴스프라임 처음 집 사면 LTV 80%…하반기 달라지는 정책은', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3, version=0, score=0.3041949210084548, payload={'제목': '오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None)]


In [18]:
print(result.points)

[ScoredPoint(id=111, version=0, score=0.3627533353944139, payload={'제목': '월 10만원 저축하면 정부가 10만원 더…18일부터 청년내일저축계좌 모집', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=51, version=0, score=0.3620797713274341, payload={'제목': '은행 가계대출 반년간 9조 넘게 줄었다', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=120, version=0, score=0.34174196642508886, payload={'제목': '문과생도 IT 인재로 육성…취업률 59% ‘눈길’', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=130, version=0, score=0.3183656206335821, payload={'제목': '뉴스프라임 처음 집 사면 LTV 80%…하반기 달라지는 정책은', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3, version=0, score=0.3041949210084548, payload={'제목': '오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은', '카테고리': '경제'}, vector=None, shard_key=None, order_value=None)]


In [20]:
for point in result.points:
    print(point.id)
    print(docs[point.id])
    print(point.payload)

111
보건복지부 제공 보건복지부는 저소득 청년에 저축액의 최대 3배까지 추가 적립해주는 청년내일저축계좌 가입자를 오는 18일부터 모집한다고 밝혔다 해당 제도는 월 10만원을 저축하면 정부가 지원금 월 10만원을 추가 적립하는 방식으로 3년간 지원해 청년의 자산 형성을 도와주는 청년특별대책제도다 참여자는 3년 만기 시 본인 납입액 360만원을 포함해 총 720만원과 예금이자를 수령하게 된다 청년내일저축계좌는 신청 시점 기준 만 19 34세인 청년이 대상이다 본인 소득 가구 소득 가구 재산 등 3가지 기준을 충족해야 한다 신청 당시에 근로 중이어야 하며 근로 사업소득이 50만원 초과 200만원 이하여야 한다 또한 자신이 속한 가구의 소득이 기준 중위소득 100 2022년 4인 가구 기준 512만1080원 이하여야 한다 가구 재산 기준은 지역에 따라 차등이 있다 대도시 3억5000만원 중소도시 2억원 농어촌 1억7000만원 이하인 가구가 대상이다 신청자가 기초생활수급자이거나 차상위계층 기준 중위소득 50 이하 이라면 혜택이 훨씬 크다 참여자가 10만원을 적립할 때마다 정부 지원금 30만원이 지급된다 3년 뒤 만기 때 총 1440만원의 적립금과 예금이자를 수령하게 된다 또한 가입 가능 연령이 만 15 39세로 더 넓으며 근로 사업소득기준도 면제된다 복지부는 그동안 청년층 대상 자산형성지원 사업을 해왔으나 그 대상이 수급자와 차상위계층에만 한정됐었다 그러나 이번에는 중위소득 100 이하인 청년으로 대상이 확대되면서 신청 대상이 지난해 1만8000명에서 올해 10만4000명으로 6배 증가했다 가입을 희망하는 청년은 복지로 홈페이지 www bokjiro go kr 를 통해 신청하면 된다 정부는 원활한 신청을 위해 신청 시작일인 18일부터 2주 7월 18 29일 간은 출생일 기준으로 5부제를 시행한다 5부제 기간에 신청하지 못한 경우 8월 1 5일에 생일과 무관하게 신청할 수 있다 대상자 선정 결과는 소득 재산 조사 등을 거쳐 10월 중에 발표된다 곽숙영 복지부 복지정책관

실습하기
1. 구축하고 싶은 문서 찾기
2. chromadb에 문서를 적재
3. 질문에 답하는 RAG시스템을 구축

chromadb 저장하기

In [26]:
persistant_client = chromadb.PersistentClient(path="news_chroma_db")

In [27]:
persistant_client = persistant_client.get_or_create_collection(

    name = "news_collection",
    embedding_function=embeddings,
    configuration={"hnsw" : {"space":"cosine"}}

)

In [38]:
saved_collection.upsert(
    ids=ids,
    documents=docs,
    metadata=metadata
)

NameError: name 'saved_collection' is not defined

chromadb 불러오기
- 이전에 저장한 DB 불러와서 사용


In [ ]:
load_client = chromadb.PersistentClient(path="news_chroma_db")
load_collection = load_client.get_collection(

    name="news_collection"
    embedding_function=embeddings # 정상 불러오기 여부 

)

In [ ]:
print(load_collection.count())


In [ ]:
load_collection.query(query_texts=['가계대출소식알려줘'], n_results=5)

In [ ]:
result_filtered = load_collection.query(query_texts={question}, n_results=3,
                                        where={"카테고리" : {"$eq": "경제"}})

NameError: name 'load_collection' is not defined

In [ ]:
result_filtered = load_collection.query(query_texts={question}, n_results=3,
                                        where={"카테고리" : {"$contains": "경제"}})


In [ ]:
result_filtered